# prod_fraud_batch_scoring / score

- Job ID `1054413573032453`, run ID `353407243695613`, run result `SUCCESS`, run duration 110 s
- Task: `score` (task run `847380244453163`), result `SUCCESS`, execution 106 s
- Notebook: `/Workspace/Users/victor.rodrigues@databricks.com/tech_bar/databricks-tech-bar-fraud-project/prod/files/scoring/notebooks/batch_scoring`
- Parameters: `{}`
- Exported with `databricks jobs export-run <task_run_id> --views-to-export CODE` and converted to .ipynb with the cell outputs (text outputs only).

# 05 - Batch prediction with `fraud_model@champion`

Task `score` of the `fraud_batch_scoring` job (scheduled daily, paused by default):

1. Reads the source table given by the `source_table` parameter (contract: `holder_id`, `created_at_brt`, `amount`).
2. Resolves the version behind the `champion` alias and scores with `fe.score_batch` — the features
   (batch and on-demand Feature Views) are computed from
   the **feature spec** saved with the model,
   with point-in-time correctness, as in training.
3. Writes the predictions to the target `fraud_batch_predictions` in **append** mode, with the model version,
   the source table and the scoring instant.

In [ ]:
# source_table: fully qualified name (catalog.schema.table); the job parameter with the same name feeds the widget
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("model_name", "")  # scored registered model (catalog.schema.name)
dbutils.widgets.text("catalog", "")  # target catalog/schema: destination of the predictions
dbutils.widgets.text("schema", "")
SOURCE_TABLE = dbutils.widgets.get("source_table")

MODEL_NAME = dbutils.widgets.get("model_name")
ALIAS = "champion"
OUTPUT_TABLE = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}.fraud_batch_predictions"
KEYS = ["holder_id", "created_at_brt", "amount"]

## 1. Output table
Created once (`IF NOT EXISTS`); each run appends rows.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {OUTPUT_TABLE} (
  holder_id      STRING    COMMENT 'Transaction holder',
  created_at_brt TIMESTAMP COMMENT 'Transaction instant (BRT)',
  amount         DOUBLE    COMMENT 'Transaction amount',
  prediction     DOUBLE    COMMENT 'Model prediction (1 = fraud)',
  model_version  INT       COMMENT 'Version of fraud_model used (champion alias)',
  source_table   STRING    COMMENT 'Scored source table',
  scored_at      TIMESTAMP COMMENT 'Scoring instant'
)
COMMENT 'Batch predictions of fraud_model@champion (fraud_batch_scoring job)'
""")

DataFrame[]

## 2. Scoring with `score_batch`
The alias is resolved to a version number before scoring: the version used is recorded
in the output even if `champion` changes during the run. Only the columns of the source contract are sent;
the remaining inputs come from the feature spec.

In [ ]:
import mlflow
from mlflow import MlflowClient
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql import functions as F

mlflow.set_registry_uri("databricks-uc")
version = int(MlflowClient().get_model_version_by_alias(MODEL_NAME, ALIAS).version)

fe = FeatureEngineeringClient()
source_df = spark.table(SOURCE_TABLE).select(*KEYS)
scored = fe.score_batch(model_uri=f"models:/{MODEL_NAME}/{version}", df=source_df)

2026/10/30 03:30:33 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.
2026/10/30 03:30:33 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


## 3. Write (append)

In [ ]:
(scored.select(
    *KEYS,
    F.col("prediction").cast("double").alias("prediction"),
    F.lit(version).alias("model_version"),
    F.lit(SOURCE_TABLE).alias("source_table"),
    F.current_timestamp().alias("scored_at"))
 .write.mode("append").saveAsTable(OUTPUT_TABLE))